# T49 — Highland footprints in deep time: probabilistic elevation thresholds and bias-aware DBSCAN

**Cluster F: paleo-geography and paleo-topography.**

*Contributed by:* Jianping Zhou (EarthByte Group, U. Sydney; Northwest University, Xi'an).


Reproduces **Figure 4** of Zhou, J., Farahbakhsh, E., Li, S., Liu, Y., Sun, G., Cao, X., ... & Müller, R. D. (2026). *Recurrent super highlands since 2.1 Ga reveal rhythmic coupling between deep Earth and surface evolution.* Geology.

Starting from per-sample crustal-thickness inversions converted to probabilistic paleo-elevations, this notebook reconstructs samples to a chosen time with GPlately, derives exceedance probabilities for the ≥2 / ≥3 / ≥4.5 km elevation thresholds (**P2 / P3 / P45**), applies bias-aware spatial thinning, and clusters each level with DBSCAN to outline highland footprints, plateau interiors, and extreme-elevation cores. The final map overlays elevation-coloured samples, the three DBSCAN domains, and P45 cluster centroids on a GPlately base reconstruction.

**Learning objectives**

- Convert crustal-thickness / elevation envelopes into per-sample threshold-exceedance probabilities.
- Reconstruct present-day sample positions to a paleo-time with `gplately.Points`.
- Apply support-radius weighting + aggregation thinning to reduce sampling bias *before* clustering.
- Run haversine DBSCAN at several elevation thresholds and summarise clusters as tectonic-scale objects.

**Prerequisites / runtime**

- `gplately`, `plate_model_manager`, `cartopy`, `scikit-learn`, `scipy`, `shapely`, `matplotlib`, `pandas`, `numpy`.
- First run downloads the **Cao2024** plate model via `plate_model_manager` (cached under `gplately_data/`, gitignored).
- Input table is bundled under `data/paleo_elevation_zhou/` (see *Configuration*). A single-snapshot run takes ~1–2 min after the model download.

> **Plotting note.** The rest of the suite uses pyGMT; this contribution keeps its original Cartopy / Mollweide rendering (it qualifies as a GPlately workflow). A pyGMT port is a natural follow-up — see *Extend this*.

## Configuration

In [ ]:
# === USER CONFIGURATION =====================================================
from pathlib import Path
import os as _os
# Run from repo root or from Notebooks/ — both resolve to the repo root.
if Path("../data").exists() and not Path("data").exists():
    _os.chdir("..")

import sys
import numpy as np
import pandas as pd

import gplately
from plate_model_manager import PlateModelManager

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import cartopy.crs as ccrs

from sklearn.cluster import DBSCAN
from sklearn.neighbors import BallTree
from scipy.spatial import ConvexHull
from scipy.stats import gaussian_kde

# ------------------------- Data & model ------------------------------------
# Per-sample crustal-thickness + paleo-coordinates + elevation envelopes
# (slim 42-column / 101k-row copy of the full Paleotopography Data
# Assimilation master table, bundled with the suite).
CSV_PATH   = Path("data/paleo_elevation_zhou/global_crustal_thickness_with_paleo_coords.csv")
PLATE_MODEL = "Cao2024"
PMM_CACHE   = "gplately_data"   # gitignored plate-model-manager download cache

# ------------------------- Snapshot & thresholds ---------------------------
reconstruction_time = 1750     # Ma
window              = 50       # Myr (samples with Age within [t, t+window] are kept)
prob_threshold      = 0.7      # exceedance-probability cut for a level to "count"

# ------------------------- Bias-aware clustering scales ---------------------
thin_km           = 75         # aggregation-thinning radius
support_radius_km = 300        # neighbourhood for spatial down-weighting
eps_km            = 900        # DBSCAN neighbourhood radius
min_samples       = 3          # DBSCAN core-point threshold
earth_radius_km   = 6371.0

# ------------------------- Which levels to map -----------------------------
LEVEL_FOOTPRINT = "P2_km"      # >=2 km domain
LEVEL_MID       = "P3_km"      # >=3 km mid (optional overlay)
LEVEL_CORE      = "P45_km"     # >=4.5 km core

# ------------------------- Mid-level (P3) overlay ---------------------------
SHOW_P3_LAYER  = True
SHOW_P3_LABELS = False
P3_COLOR       = "#f4a261"
P3_ALPHA       = 0.22
P3_SIZE_SCALE  = 1600
P3_EXCLUDE_P45 = False

# ------------------------- P45 visualisation modes -------------------------
SHOW_P45_POINTS    = True
SHOW_P45_INFLUENCE = False
SHOW_P45_HULLS     = True
SHOW_P45_KDE       = False

# ------------------------- Elevation dots ----------------------------------
SHOW_ELEVATION_DOTS = True
DOT_SIZE      = 90
DOT_LINEWIDTH = 0.25
E_VMIN, E_VMAX = 0.0, 6.0      # fixed elevation colour scale (km)

base_cmap = plt.cm.terrain
_colors   = base_cmap(np.linspace(0.35, 1.0, 256))
elev_cmap = LinearSegmentedColormap.from_list("highland", _colors)

# ------------------------- Figure output (gitignored) ----------------------
SAVE_FIG_DIR = Path("highland_dbscan_figs")
SAVE_FIG_DIR.mkdir(parents=True, exist_ok=True)

# derived window bounds
t_min = reconstruction_time
t_max = reconstruction_time + window
# ===========================================================================

print("Environment")
print(f"  python  {sys.version.split()[0]}")
for _m in (np, pd, gplately, mpl):
    print(f"  {_m.__name__:8s}{getattr(_m, '__version__', 'n/a')}")
import cartopy, sklearn, scipy
for _m in (cartopy, sklearn, scipy):
    print(f"  {_m.__name__:8s}{_m.__version__}")

In [ ]:
def export_plot(fig, name, dpi=300, save_svg=True, save_pdf=True, save_png=True):
    """Save a figure for publication + quick-look. Does NOT modify figure content."""
    SAVE_FIG_DIR.mkdir(parents=True, exist_ok=True)
    # editable text in Illustrator / journal-safe fonts
    mpl.rcParams["pdf.fonttype"] = 42
    mpl.rcParams["ps.fonttype"]  = 42
    mpl.rcParams["svg.fonttype"] = "none"
    if save_pdf:
        fig.savefig(SAVE_FIG_DIR / f"{name}.pdf", dpi=dpi, bbox_inches="tight")
    if save_svg:
        fig.savefig(SAVE_FIG_DIR / f"{name}.svg", dpi=dpi, bbox_inches="tight")
    if save_png:
        fig.savefig(SAVE_FIG_DIR / f"{name}.png", dpi=450, bbox_inches="tight")
    print(f"Exported: {name}")

## 1. Load the plate model and data

The Cao et al. (2024) model is fetched on first run via `plate_model_manager`
and cached under `gplately_data/`. The input table carries, per sample, the
crustal-thickness envelope (`z0_min/mid/max_km`), the isostatic elevation
envelope (`Elev_abs_low/_absolute/_high_km`), the age range (`Age_Min`,
`Age_Max`) and present-day coordinates (`Lon`, `Lat`).

In [ ]:
pm_manager    = PlateModelManager()
cao2024_model = pm_manager.get_model(PLATE_MODEL, data_dir=PMM_CACHE)

rotation_model    = cao2024_model.get_rotation_model()
topology_features = cao2024_model.get_topologies()
static_polygons   = cao2024_model.get_static_polygons()

coastlines = cao2024_model.get_layer("Coastlines")
continents = cao2024_model.get_layer("ContinentalPolygons")
COBs       = cao2024_model.get_layer("COBs")

model = gplately.PlateReconstruction(rotation_model, topology_features, static_polygons)
gplot = gplately.PlotTopologies(model, coastlines=coastlines,
                                continents=continents, COBs=COBs)

pct_data = pd.read_csv(CSV_PATH)
print(f"Loaded {len(pct_data):,} samples from {CSV_PATH}")

In [ ]:
def add_elevation_probabilities(df, n_draws=300, sampler="triangular",
                                thresholds=(2.0, 3.0, 4.5)):
    """Monte-Carlo exceedance probabilities P(elev >= thr) from the (z0, elev) envelope."""
    zmin = df["z0_min_km"].to_numpy(dtype=float)
    zmid = df["z0_mid_km"].to_numpy(dtype=float)
    zmax = df["z0_max_km"].to_numpy(dtype=float)

    E_low  = df["Elev_abs_low_km"].to_numpy(dtype=float)
    E_mid  = df["Isostatic_Elevation_absolute_km"].to_numpy(dtype=float)
    E_high = df["Elev_abs_high_km"].to_numpy(dtype=float)

    rng   = np.random.default_rng(0)
    probs = {thr: np.full(len(df), np.nan, dtype=float) for thr in thresholds}

    for i in range(len(df)):
        if not (np.isfinite(zmin[i]) and np.isfinite(zmid[i]) and np.isfinite(zmax[i])):
            continue
        if not (np.isfinite(E_low[i]) and np.isfinite(E_mid[i]) and np.isfinite(E_high[i])):
            continue

        if sampler == "uniform":
            z0 = rng.uniform(zmin[i], zmax[i], size=n_draws)
        else:
            if (zmax[i] > zmin[i]) and (zmax[i] >= zmid[i] >= zmin[i]):
                z0 = rng.triangular(zmin[i], zmid[i], zmax[i], size=n_draws)
            else:
                z0 = np.full(n_draws, zmid[i])

        elev = np.interp(z0, [zmin[i], zmid[i], zmax[i]], [E_low[i], E_mid[i], E_high[i]])
        for thr in thresholds:
            probs[thr][i] = np.mean(elev >= thr)

    df["P2_km"], df["P3_km"], df["P45_km"] = probs[2.0], probs[3.0], probs[4.5]
    return df

if not all(c in pct_data.columns for c in ["P2_km", "P3_km", "P45_km"]):
    pct_data = add_elevation_probabilities(pct_data)

## 2. Time window, reconstruction and probabilistic classification

Keep samples whose age range overlaps `[t, t+window]`, reconstruct their
present-day positions to `reconstruction_time` with `gplately.Points`, then
flag each level as *definite* (P ≥ 0.7) or *possible* (0.3 ≤ P < 0.7).

In [ ]:
# --- time-window filter ---
time_mask  = (pct_data["Age_Min"] <= t_max) & (pct_data["Age_Max"] >= t_min)
slice_data = pct_data.loc[time_mask].copy()
print(f"Time-window samples (raw): {len(slice_data)}")

# --- ensure reconstructed coords exist ---
need_recon = (
    ("rlat" not in slice_data.columns) or ("rlon" not in slice_data.columns)
    or slice_data["rlat"].isna().any() or slice_data["rlon"].isna().any()
)
if need_recon:
    gpts = gplately.Points(
        model,
        slice_data["Lon"].to_numpy(dtype=float),
        slice_data["Lat"].to_numpy(dtype=float),
    )
    rlons, rlats = gpts.reconstruct(reconstruction_time, return_array=True)
    slice_data["rlon"] = rlons
    slice_data["rlat"] = rlats

finite_recon = np.isfinite(slice_data["rlat"].to_numpy()) & np.isfinite(slice_data["rlon"].to_numpy())
slice_data   = slice_data.loc[finite_recon].copy()
print(f"Reconstructable samples: {len(slice_data)} / {time_mask.sum()}")

In [ ]:
# --- probabilistic classification ---
P2  = slice_data["P2_km"].to_numpy(float)
P3  = slice_data["P3_km"].to_numpy(float)
P45 = slice_data["P45_km"].to_numpy(float)

oro_definite  = P2 >= 0.7
oro_possible  = (P2 >= 0.3) & (P2 < 0.7)
plat_definite = P3 >= 0.7
plat_possible = (P3 >= 0.3) & (P3 < 0.7)
ext_definite  = P45 >= 0.7
ext_possible  = (P45 >= 0.3) & (P45 < 0.7)

E_mid  = slice_data["Isostatic_Elevation_absolute_km"].to_numpy(float)
E_low  = slice_data["Elev_abs_low_km"].to_numpy(float)
E_high = slice_data["Elev_abs_high_km"].to_numpy(float)
uncertainty = E_high - E_low

# marker transparency scales inversely with elevation uncertainty
alpha_all = np.clip(1 - uncertainty / 4.0, 0.45, 1.0)

## 3. Bias-aware thinning + DBSCAN

For one elevation level the pipeline (a) keeps samples above `prob_threshold`,
(b) collapses duplicate paleo-locations, (c) down-weights each point by its
neighbour count within `support_radius_km`, (d) aggregates points within
`thin_km` into representative nodes (combining their probabilities), and
(e) clusters the thinned nodes with haversine DBSCAN. Spatial scales are fixed
across P2/P3/P45 so the three domains are directly comparable.

In [ ]:
def run_biasaware_dbscan(slice_df: pd.DataFrame, level_col: str) -> pd.DataFrame:
    if level_col not in slice_df.columns:
        raise KeyError(f"Missing {level_col} in slice_data columns.")

    # probability filter
    df = slice_df.loc[slice_df[level_col].astype(float) >= prob_threshold].copy()
    print(f"Samples passing {level_col} >= {prob_threshold}: {len(df)}")

    # collapse to unique paleo-locations
    df["rlat_r"] = df["rlat"].round(3)
    df["rlon_r"] = df["rlon"].round(3)
    agg = {
        "rlat": "first", "rlon": "first",
        "Age_Min": "min", "Age_Max": "max",
        "Isostatic_Elevation_absolute_km": "mean",
        "Elev_abs_low_km": "mean", "Elev_abs_high_km": "mean",
    }
    for col in ["P2_km", "P3_km", "P45_km"]:
        if col in df.columns:
            agg[col] = "mean"
    df = df.groupby(["rlat_r", "rlon_r"]).agg(agg).reset_index(drop=True)
    print(f"Collapsed -> {len(df)} unique paleo-locations")

    # spatial weighting
    coords_all = np.radians(np.vstack([df["rlat"].to_numpy(float), df["rlon"].to_numpy(float)]).T)
    tree_all   = BallTree(coords_all, metric="haversine")
    support_rad = support_radius_km / earth_radius_km
    counts = tree_all.query_radius(coords_all, r=support_rad, count_only=True)
    df["spatial_weight"] = 1.0 / np.maximum(counts, 1)
    print(f"Mean neighbour count within {support_radius_km} km: {counts.mean():.2f}")

    # aggregation-based thinning
    thin_rad = thin_km / earth_radius_km
    visited  = np.zeros(len(df), dtype=bool)
    thin_records = []
    for i in range(len(df)):
        if visited[i]:
            continue
        idx = tree_all.query_radius(coords_all[i:i+1], r=thin_rad)[0]
        visited[idx] = True
        local = df.iloc[idx]

        P_vals = local[level_col].to_numpy(dtype=float)
        P_comb = 1.0 - np.prod(1.0 - P_vals)   # combined exceedance probability

        thin_records.append({
            "rlat": np.average(local["rlat"]),
            "rlon": np.average(local["rlon"]),
            level_col: P_comb,
            "spatial_weight": np.sum(local["spatial_weight"]),
            "Isostatic_Elevation_absolute_km": np.average(local["Isostatic_Elevation_absolute_km"]),
            "Elev_abs_low_km":  np.average(local["Elev_abs_low_km"]),
            "Elev_abs_high_km": np.average(local["Elev_abs_high_km"]),
        })
    thin_df = pd.DataFrame(thin_records)

    # diagnostic: raw points merged per retained node
    merged_counts = []
    for rec in thin_records:
        rep_coord = np.radians([[rec["rlat"], rec["rlon"]]])
        merged_counts.append(len(tree_all.query_radius(rep_coord, r=thin_rad)[0]))
    thin_df["merged_n"] = merged_counts

    print(f"After thinning to ~{thin_km} km resolution: {len(thin_df)} points")
    if merged_counts:
        print(f"Mean / median samples merged per node: "
              f"{np.mean(merged_counts):.2f} / {np.median(merged_counts):.1f}")
        print(f"Max merged: {np.max(merged_counts)} | "
              f"90th/95th pct: {np.percentile(merged_counts, 90):.0f}/"
              f"{np.percentile(merged_counts, 95):.0f}")

    # DBSCAN on thinned nodes
    coords_thin = np.radians(np.vstack([thin_df["rlat"].to_numpy(float),
                                        thin_df["rlon"].to_numpy(float)]).T)
    eps_rad = eps_km / earth_radius_km
    db = DBSCAN(eps=eps_rad, min_samples=min_samples,
                metric="haversine", algorithm="ball_tree").fit(coords_thin)
    thin_df["cluster_id"]      = db.labels_
    thin_df["cluster_id_plot"] = db.labels_

    n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    print(f"Detected clusters: {n_clusters} | noise fraction: {np.mean(db.labels_ == -1):.2f}")
    if len(thin_df) > 1:
        ttree = BallTree(coords_thin, metric="haversine")
        dist, _ = ttree.query(coords_thin, k=2)
        print(f"Mean nearest-neighbour spacing (thinned): "
              f"{np.mean(dist[:, 1]) * earth_radius_km:.1f} km")
    return thin_df

In [ ]:
print("\n=== P2 (footprint) ===")
dbscan_thin_P2 = run_biasaware_dbscan(slice_data, LEVEL_FOOTPRINT)

if P3_EXCLUDE_P45:
    slice_data_P3 = slice_data.loc[
        (slice_data[LEVEL_MID].astype(float)  >= prob_threshold) &
        (slice_data[LEVEL_CORE].astype(float) <  prob_threshold)
    ].copy()
else:
    slice_data_P3 = slice_data

print("\n=== P3 (mid) ===")
dbscan_thin_P3 = run_biasaware_dbscan(slice_data_P3, LEVEL_MID)

print("\n=== P45 (core) ===")
dbscan_thin_P45 = run_biasaware_dbscan(slice_data, LEVEL_CORE)

# P45 centroids for the star overlay
noise_P45    = dbscan_thin_P45["cluster_id_plot"] == -1
clusters_P45 = dbscan_thin_P45.loc[~noise_P45].copy()
summary_df = (
    clusters_P45.groupby("cluster_id_plot", as_index=False)
    .agg(centroid_lon=("rlon", "mean"), centroid_lat=("rlat", "mean"),
         n_points=("rlon", "size"),     mean_weight=("spatial_weight", "mean"))
    .rename(columns={"cluster_id_plot": "cluster"})
)
summary_df = summary_df[summary_df["cluster"] != -1].copy()

## 4. Map — elevation dots + P2 / P3 / P45 DBSCAN overlays + P45 centroids

Elevation-coloured samples sit on the GPlately base reconstruction; the P2
footprint, P3 mid-level and P45 core are layered as DBSCAN domains, with P45
convex-hull envelopes, ring nuclei, per-cluster labels and centroid stars.

In [ ]:
fig = plt.figure(figsize=(16, 9), dpi=150)
ax  = fig.add_subplot(111, projection=ccrs.Mollweide(central_longitude=0))
ax.set_global()
ax.gridlines(color="0.85", linestyle="--", linewidth=0.5)

gplot.time = reconstruction_time

# --- base reconstruction ---
gplot.plot_continents(ax, color="0.96", alpha=0.75, linewidth=0.25)
gplot.plot_continent_ocean_boundaries(ax, facecolor="#e8e9ea", linewidth=0.0, alpha=0.75)
gplot.plot_coastlines(ax, facecolor="#e8e9ea", linewidth=0.18, alpha=0.45)

# --- tectonic features ---
gplot.plot_ridges_and_transforms(ax, color="red",  linewidth=0.6, alpha=0.55)
gplot.plot_trenches(ax,            color="0.15",    linewidth=0.55, alpha=0.65)
gplot.plot_subduction_teeth(ax,    color="0.15",    linewidth=0.35, alpha=0.55)

# --- base dots: elevation colour-coded ---
if SHOW_ELEVATION_DOTS:
    sc = ax.scatter(
        slice_data["rlon"], slice_data["rlat"],
        c=E_mid, cmap=elev_cmap, vmin=E_VMIN, vmax=E_VMAX,
        s=DOT_SIZE, alpha=alpha_all, edgecolor="k", linewidth=DOT_LINEWIDTH,
        transform=ccrs.PlateCarree(), zorder=2,
        label="Samples coloured by elevation",
    )
    cbar = fig.colorbar(sc, ax=ax, orientation="vertical", shrink=0.55, pad=0.03)
    cbar.set_label("Isostatic elevation (km)", fontsize=8)
    cbar.ax.tick_params(labelsize=7)

# --- P2 footprint ---
clusters_P2 = dbscan_thin_P2.loc[dbscan_thin_P2["cluster_id_plot"] != -1].copy()
ax.scatter(
    clusters_P2["rlon"], clusters_P2["rlat"],
    s=clusters_P2["spatial_weight"].to_numpy() * 2500,
    c=clusters_P2["cluster_id_plot"], cmap="Greens", alpha=0.22, edgecolor="none",
    transform=ccrs.PlateCarree(), zorder=3, label="P2 ≥2 km domain",
)

# --- P3 mid-layer ---
if SHOW_P3_LAYER:
    clusters_P3 = dbscan_thin_P3.loc[dbscan_thin_P3["cluster_id_plot"] != -1].copy()
    ax.scatter(
        clusters_P3["rlon"], clusters_P3["rlat"],
        s=clusters_P3["spatial_weight"].to_numpy() * P3_SIZE_SCALE,
        facecolor=P3_COLOR, edgecolor="none", alpha=P3_ALPHA,
        transform=ccrs.PlateCarree(), zorder=4, label="P3 ≥3 km (mid)",
    )
    if SHOW_P3_LABELS:
        for cid in sorted(set(clusters_P3["cluster_id_plot"])):
            if cid == -1:
                continue
            pts = clusters_P3[clusters_P3["cluster_id_plot"] == cid]
            if len(pts) < 6:
                continue
            ax.text(pts["rlon"].mean() + 1.0, pts["rlat"].mean() + 0.6, f"M{cid}",
                    fontsize=8, weight="bold",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=P3_COLOR, lw=0.8, alpha=0.9),
                    transform=ccrs.PlateCarree(), zorder=9)

# --- P45 core: KDE field (optional) ---
if SHOW_P45_KDE and len(clusters_P45) > 3:
    xy  = np.vstack([clusters_P45["rlon"], clusters_P45["rlat"]])
    kde = gaussian_kde(xy, weights=clusters_P45["spatial_weight"].to_numpy())
    xx, yy = np.meshgrid(np.linspace(-180, 180, 240), np.linspace(-90, 90, 120))
    density = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
    ax.contourf(xx, yy, density, levels=6, cmap="Greys", alpha=0.18,
                transform=ccrs.PlateCarree(), zorder=4)

# --- P45 core: convex-hull envelopes ---
if SHOW_P45_HULLS:
    for cid in sorted(set(clusters_P45["cluster_id_plot"])):
        if cid == -1:
            continue
        pts = clusters_P45[clusters_P45["cluster_id_plot"] == cid]
        if len(pts) < 3:
            continue
        xy   = np.vstack([pts["rlon"], pts["rlat"]]).T
        poly = xy[ConvexHull(xy).vertices]
        ax.fill(poly[:, 0], poly[:, 1], facecolor="black", alpha=0.12,
                edgecolor="black", linewidth=1.0,
                transform=ccrs.PlateCarree(), zorder=5)

# --- P45 core: influence disks (optional) ---
if SHOW_P45_INFLUENCE:
    for _, row in clusters_P45.iterrows():
        ax.scatter(row["rlon"], row["rlat"], s=eps_km * 14, facecolor="black",
                   alpha=0.05, edgecolor="none",
                   transform=ccrs.PlateCarree(), zorder=6)

# --- P45 core: ring nuclei ---
if SHOW_P45_POINTS:
    ax.scatter(
        clusters_P45["rlon"], clusters_P45["rlat"],
        s=clusters_P45["spatial_weight"].to_numpy() * 900,
        facecolor="none", edgecolor="black", linewidth=1.4,
        transform=ccrs.PlateCarree(), zorder=7, label="P45 ≥4.5 km core",
    )

# --- P45 cluster labels ---
for cid in sorted(set(clusters_P45["cluster_id_plot"])):
    if cid == -1:
        continue
    pts = clusters_P45[clusters_P45["cluster_id_plot"] == cid]
    ax.text(pts["rlon"].mean() + 1.2, pts["rlat"].mean() + 0.8, f"C{cid}",
            transform=ccrs.PlateCarree(), fontsize=9, weight="bold",
            ha="left", va="bottom",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                      edgecolor="0.3", linewidth=0.6, alpha=0.9), zorder=10)

# --- P45 centroids (stars) ---
if summary_df is not None and len(summary_df) > 0:
    lon_star = ((summary_df["centroid_lon"].to_numpy(float) + 180) % 360) - 180
    lat_star = summary_df["centroid_lat"].to_numpy(float)
    m = np.isfinite(lon_star) & np.isfinite(lat_star)
    ax.scatter(lon_star[m], lat_star[m], marker="*", s=260, facecolor="red",
               edgecolor="black", linewidth=0.8, transform=ccrs.PlateCarree(),
               zorder=200, label="P45 cluster centroids")
    for _, row in summary_df.iterrows():
        lon = ((float(row["centroid_lon"]) + 180) % 360) - 180
        lat = float(row["centroid_lat"])
        if np.isfinite(lon) and np.isfinite(lat):
            ax.text(lon + 2.0, lat + 1.2, f"★{int(row['cluster'])}",
                    transform=ccrs.PlateCarree(), fontsize=10, weight="bold", color="red",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="black", lw=0.6, alpha=0.9),
                    zorder=201)

ax.legend(loc="lower left", frameon=True)
ax.set_title(
    f"Elevation-coloured reconstructed samples with P2 footprint + P3 mid + P45 core "
    f"at {reconstruction_time} Ma | window {t_max}-{t_min} Ma\n"
    f"thr={prob_threshold} | thin≈{thin_km} km | support={support_radius_km} km | "
    f"eps={eps_km} km | min_samples={min_samples}",
    fontsize=10,
)

plot_name = (f"DBSCAN_P2P3P45_with_elevationDots_centroids_thr{prob_threshold}_"
             f"{reconstruction_time}Ma_thin{thin_km}_eps{eps_km}_min{min_samples}")
fig.canvas.draw()
export_plot(fig, plot_name)
plt.show()

### Probabilistic highland support and per-cluster diagnostics

In [ ]:
def prob_mass(mask, P):
    m = np.isfinite(P)
    return np.nan if np.sum(m) == 0 else np.nansum(P[mask]) / np.nansum(P[m])

print("---- Probabilistic highland support (mass fractions) ----")
print(f"Orogenic ≥2 km support:    {prob_mass(oro_definite,  P2):.3f}")
print(f"Plateau  ≥3 km support:    {prob_mass(plat_definite, P3):.3f}")
print(f"Extreme  ≥4.5 km support:  {prob_mass(ext_definite,  P45):.3f}")

# tectonic-scale summary of each P45 cluster (max great-circle span inside cluster)
clusters = dbscan_thin_P45[dbscan_thin_P45["cluster_id"] != -1].copy()
rows = []
for cid in sorted(clusters["cluster_id"].unique()):
    pts    = clusters[clusters["cluster_id"] == cid]
    coords = np.radians(np.vstack([pts["rlat"], pts["rlon"]]).T)
    dist, _ = BallTree(coords, metric="haversine").query(coords, k=len(coords))
    rows.append({
        "cluster": cid,
        "centroid_lon": pts["rlon"].mean(), "centroid_lat": pts["rlat"].mean(),
        "n_nodes": len(pts),
        "max_span_km": np.nanmax(dist) * earth_radius_km,
        "mean_weight": pts["spatial_weight"].mean(),
    })
cluster_summary = pd.DataFrame(rows).sort_values("max_span_km", ascending=False)
out_csv = SAVE_FIG_DIR / f"P45_cluster_summary_{reconstruction_time}Ma.csv"
cluster_summary.to_csv(out_csv, index=False)
print(f"\nSaved cluster summary -> {out_csv}")
cluster_summary

## 5. (Optional) Assemble an animation from a time sweep

If you re-run the map cell across a series of `reconstruction_time` values
(writing one PNG per frame into `highland_dbscan_figs/`), the helper below
stitches them into an MP4 with FFmpeg. The video lands in `videos/`
(gitignored); a single snapshot is skipped automatically.

In [ ]:
import re, subprocess

MOVIE_GLOB = (f"DBSCAN_P2P3P45_with_elevationDots_centroids_thr{prob_threshold}_"
              f"*Ma_thin{thin_km}_eps{eps_km}_min{min_samples}.png")
MOVIE_FPS  = 1   # frames per second

def assemble_movie(frame_dir=SAVE_FIG_DIR, pattern=MOVIE_GLOB, fps=MOVIE_FPS):
    out_dir = Path("videos"); out_dir.mkdir(parents=True, exist_ok=True)
    out_mp4 = out_dir / "DBSCAN_evolution.mp4"

    files = list(Path(frame_dir).glob(pattern))
    if len(files) < 2:
        print(f"[movie] {len(files)} frame(s) match; need >=2. Skipping "
              f"(run the map cell over several times first).")
        return None

    def age(fp):
        m = re.search(r"_(\d+)Ma_", fp.name)
        if not m:
            raise ValueError(f"Could not parse age from {fp.name}")
        return int(m.group(1))

    files_sorted = sorted(files, key=age, reverse=True)  # old -> young
    print(f"[movie] {len(files_sorted)} frames: "
          f"{files_sorted[0].name} ... {files_sorted[-1].name}")

    concat = out_dir / "ffmpeg_input.txt"
    concat.write_text("".join(f"file '{fp.resolve().as_posix()}'\n" for fp in files_sorted))
    subprocess.run([
        "ffmpeg", "-y", "-r", str(fps), "-f", "concat", "-safe", "0",
        "-i", str(concat), "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2",
        "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "18", str(out_mp4),
    ], check=True)
    print(f"[movie] wrote {out_mp4}")
    return out_mp4

assemble_movie()

## Extend this

- **Port the map to pyGMT.** Re-draw the base reconstruction with
  `pygmt.Figure` + `gplately` feature plotting, the elevation dots with
  `fig.plot(..., cmap=True)`, and the DBSCAN hulls with `fig.plot(..., close=True)`;
  add the in-frame `{time} Ma` stamp as the last layer per house style.
- **Sweep through deep time.** Loop `reconstruction_time` over, e.g.,
  2100→0 Ma in 50-Myr steps, save one frame each, then run *Section 5* to
  animate the rhythmic appearance and decay of super-highlands.
- **Tune the thresholds.** Vary `prob_threshold`, `eps_km` and `thin_km` to
  test how robust the P2/P3/P45 footprints are to the clustering scale.
- **Swap the plate model.** Compare highland reconstructions under a different
  `PLATE_MODEL` available through `plate_model_manager`.